# 부스팅(Boosting)
- 기본적으로는 랜덤 포래스트와 유사한다.
- 다른 점은 각 트리들이 던지는 답의 오차들을 보정할 수 있도록 오차 함수가 존재한다.
- 이를 통해 성능을 끌어 올린다.
- 너어어어어어어어어어어어어어어무 오래걸려요.ㅠㅠ(차라리 딥러닝을 쓰고 말지.ㅠㅠㅠㅠ)

### 라이브러리 설치
- pip install xgboost
- pip install lightgbm


In [1]:
# 기본

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

# 혼동행렬
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
import shap

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install shap

### 분류

In [4]:
# 데이터 준비
train_df = pd.read_parquet('/content/drive/MyDrive/공유문서함/all_df07VIF끝.parquet')
test_df = pd.read_parquet('/content/drive/MyDrive/test데이터/test07.parquet')
target_df = pd.read_csv('/content/drive/MyDrive/Segment.csv')

display(train_df.columns)
display(test_df.columns)

Index(['이용카드수_신용체크', '_2순위카드이용금액', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '이용건수_신용_R12M', '쇼핑_도소매_이용금액', '연체입금원금_B0M', '쇼핑_마트_이용금액',
       '쇼핑_슈퍼마켓_이용금액', '연체입금원금_B5M', '연체입금원금_B2M', '이용금액_페이_온라인_B0M',
       '_1순위쇼핑업종_이용금액', '연속유실적개월수_기본_24M_카드', '청구금액_R6M', '월중평잔_일시불_B0M',
       '이용금액대', '이용가능여부_해외겸용_본인', '상향가능한도금액', '상향가능CA한도금액', '할인건수_R3M',
       '인입횟수_ARS_R6M', '수신거부여부_TM', '수신거부여부_메일', 'Cluster'],
      dtype='object')

Index(['기준년월', 'ID', '이용가능여부_해외겸용_본인', '보유여부_해외겸용_본인', '수신거부여부_TM',
       '수신거부여부_메일', '수신거부여부_DM', '이용금액_R3M_신용체크', '이용금액_R3M_신용', '_1순위카드이용금액',
       '이용카드수_신용체크', '_2순위카드이용금액', '_1순위카드이용건수', '_2순위카드이용건수', '이용카드수_신용',
       '상향가능한도금액', '상향가능CA한도금액', '정상청구원금_B5M', '정상청구원금_B0M', '정상청구원금_B2M',
       '이용금액_일시불_R12M', '이용금액_일시불_B0M', '이용금액_오프라인_B0M', '이용금액_일시불_R6M',
       '이용금액_일시불_R3M', '정상입금원금_B5M', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '이용금액_오프라인_R6M', '정상입금원금_B2M', '_3순위업종_이용금액', '_2순위업종_이용금액',
       '이용건수_신용_R12M', '_2순위쇼핑업종_이용금액', '최대이용금액_일시불_R12M', '이용건수_신판_R12M',
       '이용건수_일시불_R12M', '_1순위업종_이용금액', '_3순위쇼핑업종_이용금액', '이용가맹점수',
       '이용건수_오프라인_B0M', '이용건수_오프라인_R6M', '이용건수_오프라인_R3M', '쇼핑_도소매_이용금액',
       '이용건수_신용_R6M', '이용건수_신용_B0M', '이용건수_신용_R3M', '이용건수_신판_R6M',
       '이용건수_신판_B0M', '이용건수_신판_R3M', '이용건수_일시불_R6M', '이용건수_일시불_B0M',
       '이용건수_일시불_R3M', '_1순위교통업종_이용금액', '연체입금원금_B0M', '쇼핑_마트_이용금액',
       '쇼핑_슈퍼마켓_이용금액', '교통_주유이용금액', '이용금액_온라인_B0M', '연체입금원금_B5M', '연체입금원금_B2

In [5]:
train_df = train_df[['이용카드수_신용체크', '_2순위카드이용금액', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '이용건수_신용_R12M', '쇼핑_도소매_이용금액', '연체입금원금_B0M', '쇼핑_마트_이용금액',
       '쇼핑_슈퍼마켓_이용금액', '연체입금원금_B5M', '연체입금원금_B2M', '이용금액_페이_온라인_B0M',
       '_1순위쇼핑업종_이용금액', '연속유실적개월수_기본_24M_카드', '청구금액_R6M', '월중평잔_일시불_B0M',
       '이용금액대', '이용가능여부_해외겸용_본인', '상향가능한도금액', '상향가능CA한도금액', '할인건수_R3M',
       '인입횟수_ARS_R6M', '수신거부여부_TM', '수신거부여부_메일']]

In [6]:
test_df = test_df[['이용카드수_신용체크', '_2순위카드이용금액', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '이용건수_신용_R12M', '쇼핑_도소매_이용금액', '연체입금원금_B0M', '쇼핑_마트_이용금액',
       '쇼핑_슈퍼마켓_이용금액', '연체입금원금_B5M', '연체입금원금_B2M', '이용금액_페이_온라인_B0M',
       '_1순위쇼핑업종_이용금액', '연속유실적개월수_기본_24M_카드', '청구금액_R6M', '월중평잔_일시불_B0M',
       '이용금액대', '이용가능여부_해외겸용_본인', '상향가능한도금액', '상향가능CA한도금액', '할인건수_R3M',
       '인입횟수_ARS_R6M', '수신거부여부_TM', '수신거부여부_메일']]

In [7]:
target_df['Segment'].value_counts()

,count
Segment,
E,1922052
D,349242
C,127590
A,972
B,144


In [8]:
target_df = target_df.query('기준년월 ==201807')

In [9]:
encoder1 = LabelEncoder()
Seg = encoder1.fit_transform(target_df['Segment'])


In [10]:
Seg

array([3, 4, 2, ..., 2, 4, 4])

In [11]:
# 입력과 결과로 나눈다.
X = train_df
y = Seg

In [12]:
from sklearn.model_selection import train_test_split

# 전체 데이터: X, 정답 레이블: y
X_train, X_val, y_train, y_val = train_test_split(
    X, y,              # 데이터와 정답
    test_size=0.2,     # 검증용 데이터 비율 (20%)
    stratify=y,        # 클래스 비율 유지
    random_state=42    # 결과 재현을 위한 시드
)

In [13]:
X_test = test_df

In [14]:
train_X_df = pd.DataFrame(X_train)
train_y_df = pd.DataFrame(y_train)

### 기본 모델 사용하기

In [15]:
print(pd.Series(y_train).unique())

[4 3 2 0 1]


In [16]:
print(pd.Series(y_train).value_counts())

4    256274
3     46565
2     17012
0       130
1        19
Name: count, dtype: int64


In [17]:
model5 = LGBMClassifier(device='gpu', verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, X_train, y_train, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

평균 f1 Score : 0.859634375


In [18]:
model6 = XGBClassifier(tree_method='gpu_hist', predictor='gpu_predictor',
                       n_jobs=-1, verbosity=0, use_label_encoder=False)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

r2 = cross_val_score(model6, X_train, y_train, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean()}')

평균 f1 Score : 0.8670687499999999


In [19]:
10/0

ZeroDivisionError: division by zero

| 클래스 | 샘플 수    | 기본 가중치(역수) | 완화된 가중치(로그) | 추천 가중치 (예시) |
|--------|------------|-------------------|---------------------|--------------------|
| 4      | 1,922,052  | 0.25              | 0.22                | 1                  |
| 3      | 349,242    | 1.37              | 0.95                | 1.5                |
| 2      | 127,590    | 3.76              | 1.5                 | 4                  |
| 0      | 972        | 493.9             | 6.2                 | 50                 |
| 1      | 144        | 3330.5            | 7.8                 | 50                 |


In [31]:
### 가중치 줘보기
class_weights = {0: 150, 1: 50, 2: 4, 3: 1.5, 4: 1}

In [34]:
model3 = LGBMClassifier(device='gpu', class_weight=class_weights, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r3 = cross_val_score(model3, X_train, y_train, scoring='f1_weighted', cv=kfold)
print(f'평균 f1 Score : {r3.mean()}')

평균 f1 Score : 0.851900503987433


In [32]:
model4 = XGBClassifier(tree_method='gpu_hist', predictor='gpu_predictor',
                       n_jobs=-1, verbosity=0, use_label_encoder=False)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

# 클래스별 가중치 예시

f1_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]   # .iloc → []

    sample_weights = np.array([class_weights[label] for label in y_tr])

    model4.fit(X_tr, y_tr, sample_weight=sample_weights)
    preds = model4.predict(X_val)

    f1 = f1_score(y_val, preds, average='weighted')
    f1_scores.append(f1)
    print(f'Fold {fold+1} F1 Score: {f1:.4f}')


Fold 1 F1 Score: 0.8548
Fold 2 F1 Score: 0.8550
Fold 3 F1 Score: 0.8531
Fold 4 F1 Score: 0.8547
Fold 5 F1 Score: 0.8561
Fold 6 F1 Score: 0.8542
Fold 7 F1 Score: 0.8532
Fold 8 F1 Score: 0.8529
Fold 9 F1 Score: 0.8576
Fold 10 F1 Score: 0.8536


### 모델 하이퍼 파라미터 튜닝

In [ ]:
10/0

In [20]:
# 튜닝할 하이퍼 파라미터 후보 값
# n_estimators : 트리의 개수
# learning_rate : 학습률. 오차 보정을 위해 상수 값들을 수정하는 정도
params = {
    'n_estimators' : [250,300,350],
    'learning_rate' : [0.01, 0.05]
}
# 사용할 모델 객체를 생성한다.
model1 = LGBMClassifier(device='gpu', verbose=-1)
# 최적의 하이퍼 파라미터를 찾는다.
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
grid_clf1 = GridSearchCV(model1, param_grid=params, scoring='f1_micro', cv=kfold)
grid_clf1.fit(X_train, y_train)
print(f'최적의 하이퍼 파라미터 : {grid_clf1.best_params_}')
print(f'최적의 모델 평균 성능 : {grid_clf1.best_score_}')

최적의 하이퍼 파라미터 : {'learning_rate': 0.05, 'n_estimators': 300}
최적의 모델 평균 성능 : 0.864265625


In [21]:
# 튜닝할 하이퍼 파라미터 후보 값
# booster : 내부에서 사용할 알고리즘. gbtree - 결정트리, gblinear - 선형모델
# n_estimators : 트리의 개수
# learning_rate : 학습률. 오차 보정을 위해 상수 값들을 수정하는 정도

params = {
    'booster' : ['gbtree'],
    'n_estimators' : [400, 450, 500],
    'learning_rate' : [0.05, 0.1, 0.3]
}

model2 = XGBClassifier(tree_method='gpu_hist', predictor='gpu_predictor',
                       n_jobs=-1, verbosity=0, use_label_encoder=False)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

grid_clf2 = GridSearchCV(model2, param_grid=params, scoring='f1_micro', cv=kfold, n_jobs=-1)
grid_clf2.fit(X_train, y_train)

print(f'최적의 하이퍼 파라미터 : {grid_clf2.best_params_}')
print(f'최적의 모델 평균 성능 : {grid_clf2.best_score_:.4f}')

최적의 하이퍼 파라미터 : {'booster': 'gbtree', 'learning_rate': 0.1, 'n_estimators': 450}
최적의 모델 평균 성능 : 0.8675


### 예측을 수행한다

In [ ]:
이용금액대encoder = LabelEncoder()
test_df['이용금액대'] = 이용금액대encoder.fit_transform(test_df['이용금액대'])
할인건수encoder = LabelEncoder()
test_df['할인건수_R3M'] = 할인건수encoder.fit_transform(test_df['할인건수_R3M'])
인입횟수encoder = LabelEncoder()
test_df['인입횟수_ARS_R6M'] = 인입횟수encoder.fit_transform(test_df['인입횟수_ARS_R6M'])

In [ ]:
# 예측할 데이터를 읽어온다.
df1 = test_df.copy()
df2 = df1.copy()
df3 = df2.copy()
df4 = df3.copy()
df5 = df4.copy()
df6 = df5.copy()
df1

In [ ]:
y_pred1 = grid_clf1.predict(df1)
y_pred2 = grid_clf2.predict(df2)

In [ ]:
model4

In [ ]:
model4.fit(X_train, y_train)

In [ ]:
model5

In [ ]:
model5.fit(X_train, y_train)

In [ ]:
model6

In [ ]:
model6.fit(X_train, y_train)

In [ ]:
y_pred4 = model4.predict(df4)

In [ ]:
model5.fit(X_train, y_train)
y_pred5= model5.predict(df5)

In [ ]:
model6.fit(X_train, y_train)
y_pred6= model6.predict(df6)

In [ ]:
y_true = target_df

In [35]:
# 3. 검증셋 예측
y_pred = model3.predict(X_val)

# 4. 성능 평가
print("=== Classification Report ===")
print(classification_report(y_val, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_val, y_pred))

NotFittedError: Estimator not fitted, call fit before exploiting the model.

## 혼동행렬

In [ ]:
예측용결과40만개 = y_true['Segment']
예측용결과40만개.value_counts()

In [ ]:
예측용결과40만개

In [ ]:
# 혼동행렬을 위한 새로운 데이터프레임 생성
예측용데이터40만개 = train_X
예측용데이터40만개['Segment'] = 예측용결과40만개
# 인코딩
예측용인코더 = LabelEncoder()
y_true = 예측용인코더.fit_transform(예측용데이터40만개['Segment'])

In [ ]:
# 예측용 데이터와 예측용 결과데이터로 나눠준다
혼동X = 예측용데이터40만개.drop('Segment', axis=1)

In [ ]:
y_pred7 = grid_clf1.predict(혼동X)
y_pred8 = grid_clf2.predict(혼동X)
y_pred9 = model4.predict(혼동X)
y_pred10 = model5.predict(혼동X)
y_pred11 = model6.predict(혼동X)

In [ ]:
result_data1 = encoder1.inverse_transform(y_pred1)
result_data2 = encoder1.inverse_transform(y_pred2)
result_data4 = encoder1.inverse_transform(y_pred4)
result_data5 = encoder1.inverse_transform(y_pred5)
result_data6 = encoder1.inverse_transform(y_pred6)

In [ ]:
result_data7 = 예측용인코더.inverse_transform(y_pred7)
result_data8 = 예측용인코더.inverse_transform(y_pred8)
result_data9 = 예측용인코더.inverse_transform(y_pred9)
result_data10 = 예측용인코더.inverse_transform(y_pred10)
result_data11 = 예측용인코더.inverse_transform(y_pred11)

In [ ]:
result_data7

In [ ]:
values, counts = np.unique(result_data7, return_counts=True)

# 보기 좋게 정리
for val, count in zip(values, counts):
    print(f"{val}: {count}")

In [ ]:
values, counts = np.unique(result_data6, return_counts=True)

# 보기 좋게 정리
for val, count in zip(values, counts):
    print(f"{val}: {count}")

In [ ]:
result_data7

In [ ]:
y_true = 예측용인코더.inverse_transform(y_true)

In [ ]:
y_true

In [ ]:
result_data5
result_data6

In [ ]:
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)


In [ ]:
# 예측 결과 (y_pred), 실제 정답 (y_true)
cm = confusion_matrix(y_true, result_data7, labels=['A', 'B', 'C', 'D', 'E'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'B', 'C', 'D', 'E'])

plt.figure(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# 예측 결과 (y_pred), 실제 정답 (y_true)
cm = confusion_matrix(y_true, result_data8, labels=['A', 'B', 'C', 'D', 'E'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'B', 'C', 'D', 'E'])

plt.figure(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 예측 결과 (y_pred), 실제 정답 (y_true)
cm = confusion_matrix(y_true, result_data9, labels=['A', 'B', 'C', 'D', 'E'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'B', 'C', 'D', 'E'])

plt.figure(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 예측 결과 (y_pred), 실제 정답 (y_true)
cm = confusion_matrix(y_true, result_data10, labels=['A', 'B', 'C', 'D', 'E'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'B', 'C', 'D', 'E'])

plt.figure(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 예측 결과 (y_pred), 실제 정답 (y_true)
cm = confusion_matrix(y_true, result_data11, labels=['A', 'B', 'C', 'D', 'E'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['A', 'B', 'C', 'D', 'E'])

plt.figure(figsize=(6, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
print(classification_report(y_true, result_data11))

In [ ]:
print(classification_report(y_true, result_data8))

In [ ]:
print(classification_report(y_true, result_data9))

In [ ]:
# 예측 결과데이터를 만들기
submission_df = pd.read_parquet('/content/drive/MyDrive/test데이터/신용카드데이터_test.parquet')
submission_df1 = submission_df[-100000:]
submission_df2 = submission_df[-100000:]
submission_df3 = submission_df[-100000:]

In [ ]:
submission_df2 = submission_df2[:100000]

In [ ]:
result_data2

In [ ]:
result_data4
result_data6

In [ ]:
submission_df2

In [ ]:

submission_df1 = submission_df1[['ID','기준년월']]
submission_df2 = submission_df2[['ID','기준년월']]
submission_df3 = submission_df3[['ID','기준년월']]

submission_df1['Segment'] = result_data2
submission_df2['Segment'] = result_data4
submission_df3['Segment'] = result_data6

In [ ]:
submission_df2['Segment'].value_counts()


In [ ]:
submission_df1['Segment'].value_counts()

In [ ]:
submission_df3['Segment'].value_counts()

In [ ]:
submission_df1.drop('기준년월', axis=1, inplace=True)
submission_df2.drop('기준년월', axis=1, inplace=True)
submission_df3.drop('기준년월', axis=1, inplace=True)

In [ ]:
submission_df1.to_csv('/content/drive/MyDrive/공유문서함/XGBhyper.csv', index=False)
submission_df2.to_csv('/content/drive/MyDrive/공유문서함/XGB가중치.csv', index=False)
submission_df3.to_csv('/content/drive/MyDrive/공유문서함/XGBbasic.csv', index=False)

In [ ]:
10/0

In [ ]:
import joblib
# 전처리기와 모델을 하나의 딕셔너리에 묶기
bundle = {
    'encoder': encoder1,
    'models': {
        'xgb_basic': model6,
        'lgbm_basic': model5,
        'lgbm_tuned': grid_clf1.best_estimator_,
        'xgb_tuned': grid_clf2.best_estimator_,
        'xgb_weight': model4
    },
    'params': {
        'xgb_basic': model6.get_params(),
        'lgbm_tuned': grid_clf1.best_params_,
        'xgb_tuned': grid_clf2.best_params_
    },
    'feature_names': train_X.columns.tolist()
}

# 저장
joblib.dump(bundle, '/content/drive/MyDrive/model/bundle1.pkl')

In [ ]:
import pickle
# 전처리기와 모델을 하나의 딕셔너리에 묶기
bundle = {
    'encoder': encoder1,
    'models': {
        'xgb_basic': model6,
        'lgbm_basic': model5,
        'lgbm_tuned': grid_clf1.best_estimator_,
        'xgb_tuned': grid_clf2.best_estimator_,
        'xgb_weight': model4
    },
    'params': {
        'xgb_basic': model6.get_params(),
        'lgbm_tuned': grid_clf1.best_params_,
        'xgb_tuned': grid_clf2.best_params_
    },
    'feature_names': train_X.columns.tolist()
}

# 저장
with open('/content/drive/MyDrive/model/bundle1_1.pkl', 'wb') as f:
    pickle.dump(bundle, f)

In [ ]:
test_df = pd.read_parquet('/content/drive/MyDrive/test데이터/test12.parquet')

In [ ]:
test_df = test_df[['이용카드수_신용체크', '_2순위카드이용금액', '정상입금원금_B0M', '이용금액_오프라인_R3M',
       '이용건수_신용_R12M', '쇼핑_도소매_이용금액', '연체입금원금_B0M', '쇼핑_마트_이용금액',
       '쇼핑_슈퍼마켓_이용금액', '연체입금원금_B5M', '연체입금원금_B2M', '이용금액_페이_온라인_B0M',
       '_1순위쇼핑업종_이용금액', '연속유실적개월수_기본_24M_카드', '청구금액_R6M', '월중평잔_일시불_B0M',
       '이용금액대', '이용가능여부_해외겸용_본인', '상향가능한도금액', '상향가능CA한도금액', '할인건수_R3M',
       '인입횟수_ARS_R6M', '수신거부여부_TM', '수신거부여부_메일']]

In [ ]:
test_X = test_df

In [ ]:
이용금액대encoder = LabelEncoder()
test_df['이용금액대'] = 이용금액대encoder.fit_transform(test_df['이용금액대'])
할인건수encoder = LabelEncoder()
test_df['할인건수_R3M'] = 할인건수encoder.fit_transform(test_df['할인건수_R3M'])
인입횟수encoder = LabelEncoder()
test_df['인입횟수_ARS_R6M'] = 인입횟수encoder.fit_transform(test_df['인입횟수_ARS_R6M'])

In [ ]:
# 예측할 데이터를 읽어온다.
df1 = test_df.copy()
df2 = df1.copy()
df3 = df2.copy()
df4 = df3.copy()
df5 = df4.copy()
df6 = df5.copy()
df1

In [ ]:
y_pred1 = grid_clf1.predict(df1)
y_pred2 = grid_clf2.predict(df2)

In [ ]:
result_data1 = encoder1.inverse_transform(y_pred1)
result_data2 = encoder1.inverse_transform(y_pred2)

In [ ]:
# 예측 결과데이터를 만들기
submission_df = pd.read_parquet('/content/drive/MyDrive/test데이터/신용카드데이터_test.parquet')
submission_df1 = submission_df[-100000:]
submission_df2 = submission_df[-100000:]
submission_df3 = submission_df[-100000:]

In [ ]:

submission_df1 = submission_df1[['ID','기준년월']]

submission_df1['Segment'] = result_data2


In [ ]:
submission_df1['Segment'].value_counts()

In [ ]:
submission_df1.drop('기준년월', axis=1, inplace=True)

In [ ]:
submission_df1

In [ ]:
submission_df1.to_csv('/content/drive/MyDrive/공유문서함/XGBhyper12.csv', index=False)